In [1]:
using Plots
using LinearAlgebra
using SparseArrays

# --- パラメータ設定 ---
const ħ = 1.0       # プランク定数 (簡略化)
const m = 1.0       # 質量
const L = 200.0     # 空間の広さ
const N = 400       # グリッド数
const dx = L / N    # 空間刻み
const dt = 0.5      # 時間刻み
const T_steps = 400 # 時間ステップ数

# 空間グリッド
x = range(-L/2, L/2, length=N)

# --- ポテンシャル V(x) の作成 ---
# 中央に壁を作る（トンネル効果を見るため）
V = zeros(N)
barrier_width = 10.0
barrier_height = 1.5
for i in 1:N
    if abs(x[i]) < barrier_width / 2
        V[i] = barrier_height
    end
end

# --- 初期状態: ガウス波束 ---
# 左側から壁に向かって進む波束
x0 = -50.0          # 初期位置
k0 = 2.0            # 平均波数（運動量）
σ = 10.0            # 波束の幅
ψ = @. exp(-0.5 * ((x - x0)/σ)^2) * exp(im * k0 * x)
# 正規化（確率の総和を1にする）
ψ /= norm(ψ) * sqrt(dx)

# --- ハミルトニアン行列の構築 ---
# 運動エネルギー項: (-ħ²/2m) * ∂²/∂x² を差分化
# 対角成分: 2, 非対角成分: -1
coeff = -ħ^2 / (2 * m * dx^2)
diag_elem = fill(-2 * coeff, N) .+ V  # 運動エネルギー(対角) + ポテンシャル
off_diag = fill(coeff, N-1)           # 運動エネルギー(非対角)

# スパース行列（三重対角行列）として構築
H = Tridiagonal(off_diag, diag_elem, off_diag)

# --- 時間発展演算子の作成 (クランク・ニコルソン法) ---
# (I + i*dt/2ħ * H) ψ(t+dt) = (I - i*dt/2ħ * H) ψ(t)
# A * ψ_new = B * ψ_old  =>  ψ_new = A \ (B * ψ_old)

# 単位行列
Id = I(N) 
factor = im * dt / (2 * ħ)

A = Id + factor * H
B = Id - factor * H

# Aは定数行列なので、事前にLU分解しておくと高速化できるが、
# Juliaの '\' 演算子はTridiagonalに対して最適化されているためそのままでOK

# --- アニメーションの生成 ---
println("シミュレーションを開始します...")

anim = @animate for t in 1:T_steps
    # 時間発展の計算
    global ψ = A \ (B * ψ)
    
    # 確率密度 |ψ|²
    prob = abs2.(ψ)
    
    # プロット
    plot(x, prob, 
        label="Probability density |ψ|²", 
        lw=2, color=:blue, fill=(0, 0.3, :blue),
        ylim=(0, 0.15), xlim=(-100, 100),
        xlabel="Position x", ylabel="Probability",
        title="Schrödinger Equation (t = $(t))",
        legend=:topright
    )
    
    # ポテンシャルの表示（スケール調整して重ねる）
    plot!(x, V .* 0.05, 
        label="Potential V(x) (scaled)", 
        color=:red, lw=2, linestyle=:dash
    )

    if t % 50 == 0
        println("Step: $t / $T_steps")
    end
end

# --- 保存 ---
gif(anim, "schrodinger_tunneling.gif", fps = 30)
println("保存完了: schrodinger_tunneling.gif")

シミュレーションを開始します...
Step: 50 / 400
Step: 100 / 400
Step: 150 / 400
Step: 200 / 400
Step: 250 / 400
Step: 300 / 400
Step: 350 / 400
Step: 400 / 400
保存完了: schrodinger_tunneling.gif


[ Info: Saved animation to /home/jovyan/work/schrodinger_tunneling.gif
